# C1 · Colab set-up check

Run this at the start of the project on Colab, and again whenever something looks wrong. Every check prints
PASS, WARN or FAIL, and the notebook keeps going so you see all problems in one run.

1. GPU (a warning only: G1 also runs on a CPU, just slower)
2. oodlab's and reneg's unit tests
3. the feature cache and the text bank
4. CLIP zero-shot accuracy from the cache (catches label or cache mix-ups)
5. WordNet (for the subtree groups and SSB-hard class names)
6. the CLIP text encoder reproduces the text bank
7. TANL re-run on Four-OOD matches the Kaggle run from notebook 02

**Runtime:** GPU (*Runtime → Change runtime type → T4 or L4*). About 10–20 minutes, mostly check 7.

In [ ]:
# ── Colab set-up (the same cell in every ReNeg notebook) ─────────────────────────────
import os, sys
DRIVE_ROOT = "/content/drive/MyDrive/ReNeg"          # change only if your project folder is elsewhere
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:                                    # running outside Colab (tests)
    IN_COLAB = False
    DRIVE_ROOT = os.environ.get("RENEG_DRIVE_ROOT", os.path.abspath("ReNeg"))
HOME = os.environ.get("RENEG_OODLAB_HOME", "/content/oodlab_home" if IN_COLAB else os.path.abspath("oodlab_home"))
for d in (f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/scratch", f"{HOME}/input"):
    os.makedirs(d, exist_ok=True)
if not os.path.islink(f"{HOME}/working"):              # working/ lives on Drive, scratch/ stays local
    if os.path.isdir(f"{HOME}/working") and not os.listdir(f"{HOME}/working"):
        os.rmdir(f"{HOME}/working")
    if not os.path.exists(f"{HOME}/working"):
        os.symlink(f"{DRIVE_ROOT}/oodlab_working", f"{HOME}/working")
os.environ["OODLAB_HOME"] = HOME
# a newer ReNeg_Colab_bundle*.zip in uploads/ is installed here, so a new bundle needs no C0 re-run
import glob, re, shutil, zipfile
def _reneg_version(text):
    m = re.search(r'__version__ = "([0-9.]+)"', text or "")
    return tuple(int(x) for x in m.group(1).split(".")) if m else None
_init = f"{DRIVE_ROOT}/code/reneg_code/reneg/__init__.py"
_installed = _reneg_version(open(_init).read()) if os.path.isfile(_init) else None
_newest = None
for _zp in glob.glob(f"{DRIVE_ROOT}/uploads/*.zip"):
    try:
        with zipfile.ZipFile(_zp) as _z:
            _hit = [n for n in _z.namelist() if n.endswith("reneg_code/reneg/__init__.py")]
            _v = _reneg_version(_z.read(_hit[0]).decode()) if _hit else None
        if _v and (_newest is None or _v > _newest[0]):
            _newest = (_v, _zp, _hit[0][: -len("reneg_code/reneg/__init__.py")])
    except (zipfile.BadZipFile, OSError):
        pass
if _newest and (_installed is None or _newest[0] > _installed):
    _tmp = os.path.join(HOME, "scratch", "_bundle")
    shutil.rmtree(_tmp, ignore_errors=True)
    with zipfile.ZipFile(_newest[1]) as _z:
        _z.extractall(_tmp)
    _src = os.path.join(_tmp, _newest[2])
    shutil.rmtree(f"{DRIVE_ROOT}/code/reneg_code", ignore_errors=True)
    shutil.copytree(os.path.join(_src, "reneg_code"), f"{DRIVE_ROOT}/code/reneg_code")
    os.makedirs(f"{DRIVE_ROOT}/notebooks", exist_ok=True)
    for _f in os.listdir(os.path.join(_src, "notebooks")):
        if _f.endswith(".ipynb") and not _f.startswith("C0"):
            shutil.copy2(os.path.join(_src, "notebooks", _f), f"{DRIVE_ROOT}/notebooks/{_f}")
    _fmt = lambda v: ".".join(map(str, v)) if v else "none"
    print(f"reneg code updated {_fmt(_installed)} -> {_fmt(_newest[0])} from uploads/{os.path.basename(_newest[1])}")
    if "reneg" in sys.modules:
        print("reneg was already imported in this session: Runtime -> Restart session, then run again")
CODE_ROOT = f"{DRIVE_ROOT}/code/oodlab_code"           # same name as in the Kaggle notebooks
for p in (CODE_ROOT, f"{DRIVE_ROOT}/code/reneg_code"):
    if not os.path.isdir(p):
        raise FileNotFoundError(f"{p} not found: run C0_transfer_from_kaggle first (GUIDE_COLAB.md, step 4)")
    if p not in sys.path:
        sys.path.insert(0, p)
import oodlab, reneg
print(f"oodlab {oodlab.__version__} | reneg {reneg.__version__} | project folder {DRIVE_ROOT}")

In [ ]:
from oodlab.session import start, finish, ensure_packages
ensure_packages(clip=True)
ctx = start("C1_colab_setup_check", need_gpu=False)
log = ctx.log
import subprocess, time, json
import numpy as np, pandas as pd, torch
from oodlab.report import find_cache
BACKBONE = "ViT-B/16"
WEIGHTS = f"{DRIVE_ROOT}/weights"
RUN_TANL_CHECK = True            # False skips check 7 (the slow one)
checks = []

def check(name, fn, warn_only=False):
    t0 = time.time()
    try:
        detail, status = fn(), "PASS"
    except Exception as e:
        detail, status = f"{type(e).__name__}: {e}", ("WARN" if warn_only else "FAIL")
    checks.append({"check": name, "status": status, "detail": str(detail)[:300], "seconds": round(time.time() - t0, 1)})
    print(f"[{status}] {name}: {detail}")

In [ ]:
def gpu():
    if not torch.cuda.is_available():
        raise RuntimeError("no GPU: Runtime -> Change runtime type -> T4 or L4 (G1 still runs on CPU)")
    return torch.cuda.get_device_name(0)
check("1 GPU", gpu, warn_only=True)

def run_tests(folder):
    def fn():
        r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "tests"],
                           cwd=folder, capture_output=True, text=True)
        last = (r.stdout.strip().splitlines() or ["(no output)"])[-1]
        if r.returncode != 0:
            raise RuntimeError(last + "\n" + r.stdout[-1500:])
        return last
    return fn
try:
    import pytest  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"])
check("2a oodlab tests", run_tests(CODE_ROOT))
check("2b reneg tests", run_tests(f"{DRIVE_ROOT}/code/reneg_code"))

state = {}
def cache_ok():
    c = find_cache(ctx, BACKBONE)
    b = c.load_textbank()
    s = b.summary()
    assert s["n_corpus"] == 69554 and s["n_selected"] == 10000, s
    missing = [n for n in ("imagenet_val_all", "imagenet_test", "imagenet_val", "ninco", "ssb_hard", "textures_all",
                           "textures", "inaturalist", "sun", "places", "openimage_o", "openimage_o_val") if not c.has(n)]
    assert not missing, f"missing sets: {missing}"
    state["cache"], state["bank"] = c, b
    return f"{c.dir}: {len(c.available())} sets, {s['n_corpus']} corpus words, dim {s['dim']}"
check("3 feature cache and text bank", cache_ok)

def zero_shot_acc():
    from reneg.transport import zero_shot
    fs, b = state["cache"].load_set("imagenet_val"), state["bank"]
    pred, _ = zero_shot(fs.feats, b.id_text.float(), b.logit_scale, device=ctx.device)
    acc = 100 * float((pred == fs.labels).mean())
    assert 60 <= acc <= 75, f"{acc:.1f}% is outside 60-75%"
    return f"{acc:.1f}% top-1 on the 5,000 validation images"
check("4 zero-shot accuracy", zero_shot_acc)

def wordnet():
    from reneg.wordnet import get_wordnet, wnid_name
    wn = get_wordnet()
    assert wn is not None, "nltk could not download WordNet"
    return f"n02099601 -> {wnid_name(wn, 'n02099601')}"
check("5 WordNet", wordnet, warn_only=True)

def text_encoder():
    from oodlab.clipwrap import load_clip
    from reneg.g1 import clip_text_encoder
    model = load_clip(BACKBONE, device=ctx.device, input_roots=ctx.input_roots + [WEIGHTS], download_root=WEIGHTS)
    enc = clip_text_encoder(model)
    b = state["bank"]
    cos = float((enc(list(b.id_names[:50])) * torch.nn.functional.normalize(b.id_text[:50].float(), dim=1)).sum(1).min())
    assert cos > 0.99, f"min cosine {cos:.4f}"
    return f"min cosine to the bank's ID embeddings {cos:.5f}"
check("6 CLIP text encoder", text_encoder)

### 7 · TANL on Colab matches TANL on Kaggle

Same code, same cache, same seed (0), paper settings, fp32. Small differences (≤ 0.2 points) can come from
a different GPU; larger ones mean the cache or the code differ from the Kaggle run.

In [ ]:
def tanl_matches():
    from oodlab.methods import TANL, TANLConfig
    from oodlab.runner import Runner
    c, b = state["cache"], state["bank"]
    runner = Runner(c, device=ctx.device, logger=log)
    tanl = TANL(b.id_text, b.corpus_text, b.noise_feats, TANLConfig.paper().with_(logit_scale=b.logit_scale), device=ctx.device)
    new = runner.evaluate(tanl, "four_ood", seeds=[0], config="paper")
    new.to_csv(ctx.path("tanl_four_ood_seed0.csv"), index=False)
    old_path = os.path.join(ctx.dirs.results, "02_reproduce_tanl", "tanl_runs.csv")
    if not os.path.isfile(old_path):
        raise FileNotFoundError(f"{old_path} (run C0 step 5)")
    old = pd.read_csv(old_path)
    m = old[old["protocol"] == "four_ood"] if "protocol" in old else old
    if "method" in m:
        m = m[m["method"].astype(str).str.lower().str.startswith("tanl")]
    if "seed" in m:
        m = m[m["seed"] == 0]
    if "precision" in m:
        m = m[m["precision"].astype(str) == "fp32"]
    if "config" in m:
        cfgs = [x for x in m["config"].astype(str).unique() if x.startswith("paper") and "fp16" not in x]
        m = m[m["config"].astype(str).isin(["paper"] if "paper" in cfgs else cfgs[:1])]
    both = new[["dataset", "fpr95", "auroc"]].merge(m[["dataset", "fpr95", "auroc"]], on="dataset", suffixes=("_colab", "_kaggle"))
    print(both.round(3).to_string(index=False))
    if not len(both):
        raise RuntimeError("no matching Kaggle rows (protocol four_ood, seed 0, paper, fp32): compare by eye")
    gap = float((both.fpr95_colab - both.fpr95_kaggle).abs().max())
    assert gap <= 0.2, f"largest FPR95 difference {gap:.3f} points"
    return f"largest FPR95 difference {gap:.3f} points over {len(both)} datasets"
if RUN_TANL_CHECK:
    check("7 TANL Colab == Kaggle", tanl_matches)

In [ ]:
table = pd.DataFrame(checks)
print(table.to_string(index=False))
table.to_csv(ctx.path("checks.csv"), index=False)
n_fail = int((table.status == "FAIL").sum())
finish(ctx, {"failed_checks": n_fail})
print("\nALL CHECKS PASSED -> checklist step 0.3 is Done; open G1_transport_gate.ipynb" if n_fail == 0
      else f"\n{n_fail} check(s) FAILED -> GUIDE_COLAB.md, troubleshooting")

### Finish saving to Google Drive

Colab copies files to Drive in the background. This cell waits until every file is written, so run it before
you disconnect the runtime. (It unmounts Drive: to run more cells afterwards, run the first cell again.)

In [ ]:
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print("All files are written to Google Drive. You can disconnect the runtime now.")
except ImportError:
    print("Not on Colab: nothing to flush.")